# ARC Prize 2026 — ARC-AGI-3 Agent

A Kaggle code-competition agent using the official ARC-AGI-3 framework. It tracks observed state/action outcomes, continues productive actions while they reveal new states, avoids repeating ineffective moves in the same state, and probes visible objects with coordinate actions. This remains a model-free heuristic—not a claim of human-level reasoning—but uses its action budget more deliberately than random exploration.

Internet is disabled. The ARC package is installed from the competition-provided wheel directory. The gateway creates the real `submission.parquet` during the competition rerun; a small placeholder is emitted during the notebook's validation run.

In [ ]:
# Install only from the competition's attached offline wheelhouse.
from pathlib import Path
wheelhouse = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels')
if not wheelhouse.is_dir():
    raise FileNotFoundError(f'Required competition wheelhouse not found: {wheelhouse}')
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
from __future__ import annotations

import hashlib
from collections import Counter
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class MyAgent(Agent):
    """A bounded, state-aware exploration policy for ARC-AGI-3."""

    MAX_ACTIONS = 240

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._grid_cache: dict[int, list[list[int]]] = {}
        self._state_cache: dict[int, str] = {}

    @property
    def name(self) -> str:
        return f'{super().name}.{self.MAX_ACTIONS}'

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _grid(self, frame: FrameData) -> list[list[int]]:
        cached = self._grid_cache.get(id(frame))
        if cached is not None:
            return cached
        layers = getattr(frame, 'frame', None) or []
        if not layers:
            return []
        plane = layers[0]
        grid = [list(row) for row in plane]
        self._grid_cache[id(frame)] = grid
        return grid

    def _state_key(self, frame: FrameData) -> str:
        cached = self._state_cache.get(id(frame))
        if cached is not None:
            return cached
        digest = hashlib.blake2b(digest_size=16)
        for row in self._grid(frame):
            digest.update(bytes(row))
        digest.update(str((frame.state, frame.levels_completed)).encode('utf-8'))
        key = digest.hexdigest()
        self._state_cache[id(frame)] = key
        return key

    @staticmethod
    def _action_id(action_input: Any) -> int | None:
        value = getattr(action_input, 'id', None)
        if value is None:
            return None
        return int(getattr(value, 'value', value))

    def _click_points(self, frame: FrameData) -> list[tuple[int, int]]:
        grid = self._grid(frame)
        if not grid or not grid[0]:
            return [(32, 32), (16, 16), (48, 16), (16, 48), (48, 48)]
        height, width = len(grid), len(grid[0])
        colors = Counter(value for row in grid for value in row)
        background = colors.most_common(1)[0][0]
        seen: set[tuple[int, int]] = set()
        components: list[tuple[int, int, int]] = []
        for y in range(height):
            for x in range(width):
                if grid[y][x] == background or (x, y) in seen:
                    continue
                color = grid[y][x]
                stack = [(x, y)]
                seen.add((x, y))
                sx = sy = count = 0
                while stack:
                    cx, cy = stack.pop()
                    sx += cx
                    sy += cy
                    count += 1
                    for nx, ny in ((cx - 1, cy), (cx + 1, cy), (cx, cy - 1), (cx, cy + 1)):
                        if (0 <= nx < width and 0 <= ny < height
                                and (nx, ny) not in seen and grid[ny][nx] == color):
                            seen.add((nx, ny))
                            stack.append((nx, ny))
                if count:
                    components.append((count, sx // count, sy // count))
        components.sort(reverse=True)
        points = [(min(63, x), min(63, y)) for _, x, y in components[:20]]
        if not points:
            points = [(32, 32), (16, 16), (48, 16), (16, 48), (48, 48)]
        return list(dict.fromkeys(points))

    @staticmethod
    def _legal_actions(frame: FrameData) -> list[GameAction]:
        available = getattr(frame, 'available_actions', None)
        if not available:
            return [a for a in GameAction if a is not GameAction.RESET]
        ids: set[int] = set()
        for item in available:
            raw = getattr(item, 'value', item)
            try:
                ids.add(int(raw))
            except (TypeError, ValueError):
                name = str(getattr(item, 'name', item)).upper()
                if name in GameAction.__members__:
                    ids.add(GameAction[name].value)
        return [a for a in GameAction if a.value in ids and a is not GameAction.RESET]

    def _attempt_history(self, frames: list[FrameData]) -> tuple[set[tuple[str, int, int, int]], dict[int, list[float]]]:
        attempted: set[tuple[str, int, int, int]] = set()
        rewards: dict[int, list[float]] = {}
        seen_states: set[str] = set()
        for index in range(1, len(frames)):
            before, after = frames[index - 1], frames[index]
            source_key = self._state_key(before)
            action_input = getattr(after, 'action_input', None)
            action_id = self._action_id(action_input)
            if action_id is None or action_id == GameAction.RESET.value:
                continue
            data = getattr(action_input, 'data', {}) or {}
            if hasattr(data, 'model_dump'):
                data = data.model_dump()
            x = int(data.get('x', -1)) if isinstance(data, dict) else -1
            y = int(data.get('y', -1)) if isinstance(data, dict) else -1
            attempted.add((source_key, action_id, x, y))
            before_grid, after_grid = self._grid(before), self._grid(after)
            after_key = self._state_key(after)
            reward = 1.0 if before_grid != after_grid else -0.5
            if after.levels_completed > before.levels_completed:
                reward += 12.0 * (after.levels_completed - before.levels_completed)
            if after.state is GameState.GAME_OVER:
                reward -= 1.0
            if after_key not in seen_states:
                reward += 0.25
            seen_states.add(after_key)
            rewards.setdefault(action_id, []).append(reward)
        return attempted, rewards

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        actions = self._legal_actions(latest_frame)
        if not actions:
            raise RuntimeError('The observation exposes no usable game actions.')

        state_key = self._state_key(latest_frame)
        attempted, rewards = self._attempt_history(frames)
        candidates: list[tuple[float, int, int, int, GameAction]] = []
        points = self._click_points(latest_frame)
        for order, action in enumerate(actions):
            coordinates = points if action.is_complex() else [(-1, -1)]
            past = rewards.get(action.value, [])
            average = sum(past) / len(past) if past else 0.0
            exploration = 1.0 / (1 + len(past)) ** 0.5
            score = average + exploration - order * 1e-4
            for x, y in coordinates:
                key = (state_key, action.value, x, y)
                if key not in attempted:
                    candidates.append((score, -order, -y, -x, action))
                    break
        if candidates:
            action = max(candidates, key=lambda item: item[:4])[4]
        else:
            # Every available action was tried here; use the historically most
            # productive legal action rather than cycling through hash choices.
            action = max(actions, key=lambda a: sum(rewards.get(a.value, [0.0])) / max(1, len(rewards.get(a.value, []))))
        if action.is_complex():
            tried_points = {(x, y) for key, action_id, x, y in attempted
                            if key == state_key and action_id == action.value}
            x, y = next((p for p in points if p not in tried_points), points[0])
            action.set_data({'x': x, 'y': y})
            action.reasoning = {'why': 'probe a visible object; click location not yet tried in this state'}
        else:
            action.reasoning = f'state-aware exploration: {action.name}'
        return action

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the competition gateway to accept requests.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games

    framework = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents')
    if not framework.is_dir():
        raise FileNotFoundError(f'Required ARC-AGI-3 agent framework not found: {framework}')
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /tmp/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register only the lightweight agents needed by this submission.
    agents_init = '''from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\n\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {\"random\": Random, \"myagent\": MyAgent}\n'''
    Path('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py').write_text(agents_init)
    env_text = '''SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n'''
    Path('/kaggle/working/ARC-AGI-3-Agents/.env').write_text(env_text)
    %cd /kaggle/working/ARC-AGI-3-Agents
    !MPLBACKEND=agg python main.py --agent myagent
else:
    # Kaggle validates the committed notebook before the competition rerun.
    # The gateway generates the real submission during the competition rerun.
    import pandas as pd
    validation_output = pd.DataFrame(
        [['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'],
    )
    validation_output.to_parquet('/kaggle/working/submission.parquet', index=False)
    display(validation_output)